# Error Pattern × Stimulation Windows
### Digit Span Backwards — Which windows had STIM ON during reversal-error trials?

## Error Types

| Error Type | Definition | Example (4-digit) |
|---|---|---|
| `NO_REVERSAL_exact` | Response == presented order (didn't reverse at all) | presented=1245 → resp=1245 |
| `NO_REVERSAL_partial` | First digit of response ≠ last digit of presented (wrong start) | presented=1354 → resp=5341 |
| `PARTIAL_REVERSAL` | First digit correct (started reversal) but subsequent digits wrong | presented=1245 → resp=5432 |
| `OTHER_ERROR` | Incorrect but doesn't fit above patterns |
| `CORRECT` | ACC=1 |

**Encoding:** `CRESP` = correct reversed answer (e.g. presented `1245` → CRESP `5421`).  
Presented sequence = `CRESP[::-1]`.

## Goal
For each error-type trial, identify which cognitive windows (Fixation / Stimulus / Choice / Feedback)  
had STIM ON. Show this for Session 2, Session 3, and Combined.

## Cell 1 — Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    1.2,
    'figure.dpi':        150,
    'savefig.dpi':       180,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

# Colours
C_ON  = '#1A56DB'
C_OFF = '#90A4AE'

ERROR_META = {
    'NO_REVERSAL_exact':   dict(label='No Reversal\n(exact forward)', color='#B71C1C', short='NR-exact'),
    'NO_REVERSAL_partial': dict(label='No Reversal\n(wrong start)',   color='#E64A19', short='NR-partial'),
    'PARTIAL_REVERSAL':    dict(label='Partial Reversal\n(start OK)', color='#F9A825', short='Partial'),
    'OTHER_ERROR':         dict(label='Other Error',                  color='#78909C', short='Other'),
    'CORRECT':             dict(label='Correct',                      color='#2E7D32', short='Correct'),
}

WINDOW_ORDER  = ['Fixation', 'Stimulus', 'Choice', 'Feedback']
WINDOW_COLORS = {
    'Fixation': '#7B1FA2',
    'Stimulus': '#E65100',
    'Choice':   '#1565C0',
    'Feedback': '#2E7D32',
}

STIM_THRESHOLD = 2.0

print('Imports OK')

## Cell 2 — File Paths  ← EDIT THESE

In [ ]:
JSON_PATH_S2   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1433\Report_Json_Session_Report_20260305T151703.json")
CSV_PATH_S2    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-2-Scores.csv")
EVENTS_PATH_S2 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2\Events.csv")
OUT_DIR_S2     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2")

JSON_PATH_S3   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1441\Report_Json_Session_Report_20260305T151912.json")
CSV_PATH_S3    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-3-Scores.csv")
EVENTS_PATH_S3 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3\Preprocessed Data\Events.csv")
OUT_DIR_S3     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3")

COMBINED_DIR   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Combined")
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print('Paths set.')

## Cell 3 — Core Pipeline (alignment + mA trace)

In [ ]:
def load_json_eprime(json_path, csv_path):
    with open(json_path) as f:
        report = json.load(f)
    df   = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)
    meta = dict(
        subject = str(df['Subject'].iloc[0]),
        session = str(df['Session'].iloc[0]),
        date    = str(df['SessionDate'].iloc[0]),
    )
    return report, df, meta


def build_mA_trace(report):
    ticks, mAs = [], []
    for stream in report['BrainSenseLfp']:
        for pkt in stream['LfpData']:
            ticks.append(pkt['TicksInMs'])
            mAs.append(pkt['Left']['mA'])
    ticks = np.array(ticks, dtype=float)
    mAs   = np.array(mAs,   dtype=float)
    order = np.argsort(ticks)
    return ticks[order], mAs[order]


def find_stim_anchor(report):
    for stream in report['BrainSenseLfp']:
        prev = None
        for pkt in stream['LfpData']:
            curr = pkt['Left']['mA']
            if prev is not None and prev == 0.0 and curr > 0.0:
                return pkt['TicksInMs']
            prev = curr
    raise RuntimeError('No 0→>0 mA transition found!')


def make_to_rel(report, eprime_df):
    stim_tick  = find_stim_anchor(report)
    welcome_ms = int(eprime_df['Welcome.TargetOnsetTime'].iloc[0])
    offset     = stim_tick - welcome_ms
    def to_rel(ms): return float(ms) + offset - stim_tick
    return to_rel, stim_tick


def stim_fraction(t_start, t_end, ticks_rel, mAs, threshold=STIM_THRESHOLD):
    """
    Fraction of [t_start, t_end] where mA >= threshold.
    Fallback to nearest sample when < 2 samples in window.
    """
    if t_start is None or t_end is None or t_end <= t_start:
        return np.nan
    mask = (ticks_rel >= t_start) & (ticks_rel <= t_end)
    t_r  = ticks_rel[mask]
    t_m  = mAs[mask]
    if len(t_r) < 2:
        mid = (t_start + t_end) / 2.0
        idx = int(np.argmin(np.abs(ticks_rel - mid)))
        return 1.0 if mAs[idx] >= threshold else 0.0
    dt     = np.diff(t_r)
    mid_mA = (t_m[:-1] + t_m[1:]) / 2.0
    total  = t_r[-1] - t_r[0]
    if total <= 0:
        return 1.0 if t_m[0] >= threshold else 0.0
    return float(np.sum(dt[mid_mA >= threshold]) / total)


print('Pipeline functions defined.')

## Cell 4 — Error Classification

**Encoding:** `CRESP` is the correct reversed response stored as an integer.  
Digits of `CRESP` = correct answer. `CRESP[::-1]` = presented order.

```
Example: presented = 1 2 4 5  →  correct reversal = 5 4 2 1  →  CRESP = 5421
```

**Error types:**
- `NO_REVERSAL_exact`   — resp == presented order (e.g. 1245→1245)
- `NO_REVERSAL_partial` — resp[0] ≠ presented[-1] (didn't even start the reversal)
- `PARTIAL_REVERSAL`    — resp[0] == presented[-1] but full response is wrong
- `OTHER_ERROR`         — wrong but doesn't match above
- `CORRECT`             — ACC == 1

In [ ]:
def classify_error(acc, cresp_int, resp_int, num_digits):
    """
    Classify a single trial's error type.
    cresp_int, resp_int : integer values from E-Prime (e.g. 5421, 5432)
    Returns error type string.
    """
    nd = int(num_digits)

    if acc == 1:
        return 'CORRECT'

    # Convert to zero-padded strings
    cresp = str(int(cresp_int)).zfill(nd)   # correct reversed answer
    resp  = str(int(resp_int)).zfill(nd)    # patient response
    pres  = cresp[::-1]                     # presented order

    # Error type 1: exact forward match — responded with presented order
    if resp == pres:
        return 'NO_REVERSAL_exact'

    # Error type 2: first digit of response ≠ last digit of presented
    # Patient didn't start the reversal at all
    if resp[0] != pres[-1]:
        return 'NO_REVERSAL_partial'

    # Error type 3: first digit correct (started reversal) but full response wrong
    if resp[0] == pres[-1] and resp != cresp:
        return 'PARTIAL_REVERSAL'

    return 'OTHER_ERROR'


print('classify_error defined.')

## Cell 5 — Main Loader: Build Trial × Window DataFrame

In [ ]:
def build_trial_window_df(json_path, csv_path, events_path,
                          sess_label, threshold=STIM_THRESHOLD):
    """
    Returns two DataFrames:

    1. trial_df  — one row per trial with error classification
       Columns: session, trial_num, digits, acc, cresp, resp, presented,
                error_type, stim_fix, stim_stim, stim_choice, stim_feedback
                (stim_X = True/False: was ANY sub-window of that type stimulated?)

    2. window_df — one row per sub-window instance (for detailed inspection)
       Columns: session, trial_num, error_type, window_type, sub_idx,
                t_start, t_end, stim_frac, stim_on, acc, digits
    """
    report, eprime_df, meta = load_json_eprime(json_path, csv_path)
    to_rel, stim_tick = make_to_rel(report, eprime_df)

    raw_ticks, raw_mAs = build_mA_trace(report)
    ticks_rel = raw_ticks - stim_tick

    ev = pd.read_csv(events_path, encoding='utf-8-sig', low_memory=False)

    def ev_all(etype, tn):
        return ev[(ev['Event_Type']==etype) &
                  (ev['Trial_Number']==tn)]['Time_ms'].tolist()
    def ev_first(etype, tn):
        v = ev_all(etype, tn); return v[0] if v else None

    trial_rows  = []
    window_rows = []

    trial_nums = sorted(ev['Trial_Number'].dropna().unique())

    for tn in trial_nums:
        tn = float(tn)
        r  = ev[(ev['Event_Type']=='Main Trial Start') &
                (ev['Trial_Number']==tn)]
        if r.empty: continue
        r = r.iloc[0]

        acc      = int(r['ACC'])        if pd.notna(r['ACC'])        else None
        cresp_v  = r['CRESP']           if pd.notna(r['CRESP'])      else None
        resp_v   = r['RESP']            if pd.notna(r['RESP'])       else None
        nd       = int(r['Num_Digits']) if pd.notna(r['Num_Digits']) else None

        if acc is None or cresp_v is None or resp_v is None or nd is None:
            continue

        nd_s     = str(int(cresp_v)).zfill(nd)
        presented= nd_s[::-1]
        resp_s   = str(int(resp_v)).zfill(nd)
        etype    = classify_error(acc, cresp_v, resp_v, nd)

        # ── per-digit sub-windows ─────────────────────────────────────────────
        fix_starts  = [to_rel(x) for x in ev_all('Fixation Start', tn)]
        stim_starts = [to_rel(x) for x in ev_all('Stimulus Start', tn)]
        stim_ends   = [to_rel(x) for x in ev_all('Stimulus End',   tn)]

        any_fix_on  = False
        any_stim_on = False

        # Fixation: Fixation Start[i] → Stimulus Start[i]  (~1 sec)
        for i, (fs, ss) in enumerate(zip(fix_starts, stim_starts)):
            frac = stim_fraction(fs, ss, ticks_rel, raw_mAs, threshold)
            on   = bool(frac >= 0.5) if not np.isnan(frac) else False
            if on: any_fix_on = True
            window_rows.append(dict(
                session=sess_label, trial_num=int(tn), error_type=etype,
                window_type='Fixation', sub_idx=i+1,
                t_start=fs, t_end=ss,
                stim_frac=frac, stim_on=on,
                acc=acc, digits=nd,
            ))

        # Stimulus: Stimulus Start[i] → Stimulus End[i]  (~1 sec)
        for i, (ss, se) in enumerate(zip(stim_starts, stim_ends)):
            frac = stim_fraction(ss, se, ticks_rel, raw_mAs, threshold)
            on   = bool(frac >= 0.5) if not np.isnan(frac) else False
            if on: any_stim_on = True
            window_rows.append(dict(
                session=sess_label, trial_num=int(tn), error_type=etype,
                window_type='Stimulus', sub_idx=i+1,
                t_start=ss, t_end=se,
                stim_frac=frac, stim_on=on,
                acc=acc, digits=nd,
            ))

        # Choice (single)
        cs = to_rel(ev_first('Choice Start', tn)) if ev_first('Choice Start', tn) else None
        ce = to_rel(ev_first('Choice End',   tn)) if ev_first('Choice End',   tn) else None
        frac_ch = stim_fraction(cs, ce, ticks_rel, raw_mAs, threshold)
        on_ch   = bool(frac_ch >= 0.5) if not np.isnan(frac_ch) else False
        if cs is not None:
            window_rows.append(dict(
                session=sess_label, trial_num=int(tn), error_type=etype,
                window_type='Choice', sub_idx=1,
                t_start=cs, t_end=ce,
                stim_frac=frac_ch, stim_on=on_ch,
                acc=acc, digits=nd,
            ))

        # Feedback (single)
        fs2 = to_rel(ev_first('Feedback Start', tn)) if ev_first('Feedback Start', tn) else None
        fe  = to_rel(ev_first('Feedback End',   tn)) if ev_first('Feedback End',   tn) else None
        frac_fb = stim_fraction(fs2, fe, ticks_rel, raw_mAs, threshold)
        on_fb   = bool(frac_fb >= 0.5) if not np.isnan(frac_fb) else False
        if fs2 is not None:
            window_rows.append(dict(
                session=sess_label, trial_num=int(tn), error_type=etype,
                window_type='Feedback', sub_idx=1,
                t_start=fs2, t_end=fe,
                stim_frac=frac_fb, stim_on=on_fb,
                acc=acc, digits=nd,
            ))

        trial_rows.append(dict(
            session      = sess_label,
            trial_num    = int(tn),
            digits       = nd,
            acc          = acc,
            presented    = presented,
            cresp        = nd_s,
            resp         = resp_s,
            error_type   = etype,
            stim_fix      = any_fix_on,
            stim_stim     = any_stim_on,
            stim_choice   = on_ch,
            stim_feedback = on_fb,
        ))

    trial_df  = pd.DataFrame(trial_rows)
    window_df = pd.DataFrame(window_rows)

    print(f'{sess_label}: {len(trial_df)} trials, {len(window_df)} sub-windows')
    return trial_df, window_df


print('build_trial_window_df defined.')

## Cell 6 — Load Sessions

In [ ]:
trial_s2, win_s2 = build_trial_window_df(
    JSON_PATH_S2, CSV_PATH_S2, EVENTS_PATH_S2, 'Session 2')

trial_s3, win_s3 = build_trial_window_df(
    JSON_PATH_S3, CSV_PATH_S3, EVENTS_PATH_S3, 'Session 3')

trial_all = pd.concat([trial_s2, trial_s3], ignore_index=True)
win_all   = pd.concat([win_s2,   win_s3],   ignore_index=True)

print('\nDone.')

## Cell 7 — Print: Trial-Level Error Classification Table
One row per trial — error type, presented sequence, response, and which windows had STIM ON.

In [ ]:
for sess_label, df in [('Session 2', trial_s2), ('Session 3', trial_s3)]:
    print(f'\n{"="*82}')
    print(f'  {sess_label} — Trial Error Classification + Stim Windows')
    print(f'{"="*82}')
    print(f'  {"T":>3} {"dig":>4} {"pres":>8} {"resp":>8} {"acc":>4}  '
          f'{"error_type":<22}  {"Fix":>5} {"Stim":>5} {"Choi":>5} {"Fbk":>5}')
    print(f'  {"-"*78}')
    for _, r in df.iterrows():
        def yn(v): return 'ON ' if v else 'off'
        print(f'  T{r["trial_num"]:2d} '
              f'{r["digits"]:>4}d '
              f'{r["presented"]:>8} '
              f'{r["resp"]:>8} '
              f'{r["acc"]:>4}  '
              f'{r["error_type"]:<22}  '
              f'{yn(r["stim_fix"]):>5} '
              f'{yn(r["stim_stim"]):>5} '
              f'{yn(r["stim_choice"]):>5} '
              f'{yn(r["stim_feedback"]):>5}')

## Cell 8 — Plot A: Heatmap — Error Type × Window × Stim ON/OFF

For each session: rows = trials (grouped by error type), columns = windows.  
Cell colour = STIM ON (blue) or STIM OFF (grey).

In [ ]:
def plot_trial_heatmap(trial_df, sess_label, save_path=None):
    """
    Heatmap: rows = trials sorted by error type, cols = 4 windows.
    Blue = STIM ON, grey = STIM OFF.
    Error type shown as row label colour.
    """
    # Sort trials by error type order
    etype_order = ['NO_REVERSAL_exact', 'NO_REVERSAL_partial',
                   'PARTIAL_REVERSAL', 'OTHER_ERROR', 'CORRECT']
    df = trial_df.copy()
    df['etype_rank'] = df['error_type'].map(
        {e: i for i, e in enumerate(etype_order)}).fillna(99)
    df = df.sort_values(['etype_rank', 'trial_num']).reset_index(drop=True)

    win_cols = ['stim_fix', 'stim_stim', 'stim_choice', 'stim_feedback']
    win_labels = ['Fixation', 'Stimulus', 'Choice', 'Feedback']

    n_trials = len(df)
    n_wins   = len(win_cols)

    fig, ax = plt.subplots(figsize=(7, max(4, n_trials * 0.55 + 1.5)),
                           facecolor='white')

    for ci, wc in enumerate(win_cols):
        for ri, (_, row) in enumerate(df.iterrows()):
            col = C_ON if row[wc] else C_OFF
            ax.add_patch(plt.Rectangle(
                (ci, ri), 1, 1,
                color=col, lw=0.5, edgecolor='white', zorder=2
            ))
            # ON/off text inside cell
            ax.text(ci + 0.5, ri + 0.5,
                    'ON' if row[wc] else 'off',
                    ha='center', va='center', fontsize=7.5,
                    color='white' if row[wc] else '#555',
                    fontweight='bold')

    # Row labels: trial number + error type (coloured)
    ax.set_yticks(np.arange(n_trials) + 0.5)
    ylabels = []
    ycolors = []
    for _, row in df.iterrows():
        em  = ERROR_META.get(row['error_type'], dict(short=row['error_type'], color='#333'))
        ylabels.append(f'T{row["trial_num"]:02d} ({row["digits"]}d)  {em["short"]}')
        ycolors.append(em['color'])
    ax.set_yticklabels(ylabels, fontsize=8.5)
    for ytick, color in zip(ax.get_yticklabels(), ycolors):
        ytick.set_color(color)

    ax.set_xticks(np.arange(n_wins) + 0.5)
    ax.set_xticklabels(win_labels, fontsize=10, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')

    # Colour each column header to match window colour
    for xt, wl in zip(ax.get_xticklabels(), win_labels):
        xt.set_color(WINDOW_COLORS.get(wl, '#333'))

    ax.set_xlim(0, n_wins)
    ax.set_ylim(0, n_trials)
    ax.invert_yaxis()
    ax.set_facecolor('white')
    for spine in ax.spines.values(): spine.set_visible(False)

    # Legend
    handles = [
        mpatches.Patch(color=C_ON,  label='STIM ON  (≥50% of window)'),
        mpatches.Patch(color=C_OFF, label='STIM OFF (<50% of window)'),
    ]
    ax.legend(handles=handles, fontsize=9, loc='lower right',
              bbox_to_anchor=(1.0, -0.08), ncol=2,
              framealpha=0.9, edgecolor='#ccc')

    # Separator lines between error type groups
    prev_etype = None
    for ri, (_, row) in enumerate(df.iterrows()):
        if row['error_type'] != prev_etype and ri > 0:
            ax.axhline(ri, color='#333', lw=1.5, zorder=5)
        prev_etype = row['error_type']

    ax.set_title(
        f'Stimulation Windows per Trial — {sess_label}\n'
        f'(trials sorted by error type)',
        fontsize=12, fontweight='bold', pad=28
    )
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close(fig)


print('plot_trial_heatmap defined.')

## Cell 9 — Plot A: Heatmaps

In [ ]:
plot_trial_heatmap(trial_s2, 'Session 2 (14 trials)',
                  save_path=OUT_DIR_S2 / 'error_stim_heatmap_S2.png')

plot_trial_heatmap(trial_s3, 'Session 3 (14 trials)',
                  save_path=OUT_DIR_S3 / 'error_stim_heatmap_S3.png')

plot_trial_heatmap(trial_all, 'Combined: S2 + S3 (28 trials)',
                  save_path=COMBINED_DIR / 'error_stim_heatmap_combined.png')

## Cell 10 — Plot B: Bar Chart — % of error-type trials with STIM ON per window

For each error type: what fraction of those trials had at least one stimulated
sub-window of each type?

In [ ]:
def plot_stim_rate_by_error(trial_df, sess_label, save_path=None):
    """
    For each error type × window: % of trials that had STIM ON.
    Grouped bar chart: groups = error types, bars = windows.
    """
    etype_order = [e for e in
                   ['NO_REVERSAL_exact', 'NO_REVERSAL_partial',
                    'PARTIAL_REVERSAL', 'OTHER_ERROR', 'CORRECT']
                   if e in trial_df['error_type'].unique()]

    win_cols   = ['stim_fix', 'stim_stim', 'stim_choice', 'stim_feedback']
    win_labels = ['Fixation', 'Stimulus', 'Choice', 'Feedback']

    n_e = len(etype_order)
    n_w = len(win_cols)
    bw  = 0.18
    xs  = np.arange(n_e)

    fig, ax = plt.subplots(figsize=(max(8, n_e * 2.2), 5.5), facecolor='white')

    for wi, (wc, wl) in enumerate(zip(win_cols, win_labels)):
        rates = []
        ns    = []
        for et in etype_order:
            sub = trial_df[trial_df['error_type'] == et]
            n   = len(sub)
            r   = sub[wc].sum() / n * 100 if n > 0 else 0
            rates.append(r)
            ns.append(n)

        offset = (wi - n_w/2 + 0.5) * bw
        bars = ax.bar(xs + offset, rates, width=bw,
                      color=WINDOW_COLORS[wl], alpha=0.85,
                      label=wl, zorder=3, edgecolor='white')

        for xi, (rate, n) in enumerate(zip(rates, ns)):
            if rate > 0:
                ax.text(xi + offset, rate + 1.5,
                        f'{rate:.0f}%',
                        ha='center', va='bottom', fontsize=7,
                        color=WINDOW_COLORS[wl], fontweight='bold')

    # x-axis: error type labels with n
    xlabels = []
    for et in etype_order:
        n  = len(trial_df[trial_df['error_type'] == et])
        em = ERROR_META.get(et, dict(label=et, color='#333'))
        xlabels.append(f"{em['label']}\n(n={n})")

    ax.set_xticks(xs)
    ax.set_xticklabels(xlabels, fontsize=9)
    for xt, et in zip(ax.get_xticklabels(), etype_order):
        xt.set_color(ERROR_META.get(et, dict(color='#333'))['color'])

    ax.set_ylim(0, 125)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_ylabel('% of trials with STIM ON\n(≥1 sub-window stimulated)', fontsize=11)
    ax.axhline(50, color='#ddd', lw=1, ls='--', zorder=1)
    ax.yaxis.grid(True, color='#eee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('white')

    ax.legend(title='Window', fontsize=9, title_fontsize=9,
              loc='upper right', framealpha=0.9, edgecolor='#ccc')
    ax.set_title(
        f'% of Trials with STIM ON per Window, by Error Type\n{sess_label}',
        fontsize=12, fontweight='bold'
    )
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close(fig)


print('plot_stim_rate_by_error defined.')

## Cell 11 — Plot B: Stim Rate by Error Type

In [ ]:
plot_stim_rate_by_error(trial_s2, 'Session 2 (14 trials)',
                        save_path=OUT_DIR_S2 / 'error_stim_rate_S2.png')

plot_stim_rate_by_error(trial_s3, 'Session 3 (14 trials)',
                        save_path=OUT_DIR_S3 / 'error_stim_rate_S3.png')

plot_stim_rate_by_error(trial_all, 'Combined: S2 + S3 (28 trials)',
                        save_path=COMBINED_DIR / 'error_stim_rate_combined.png')

## Cell 12 — Summary Table

In [ ]:
etype_order = ['NO_REVERSAL_exact', 'NO_REVERSAL_partial',
               'PARTIAL_REVERSAL', 'OTHER_ERROR', 'CORRECT']
win_cols    = ['stim_fix', 'stim_stim', 'stim_choice', 'stim_feedback']
win_labels  = ['Fix_ON', 'Stim_ON', 'Choice_ON', 'Feedback_ON']

rows = []
for sess_label, df in [
    ('Session 2',        trial_s2),
    ('Session 3',        trial_s3),
    ('Combined (S2+S3)', trial_all),
]:
    for et in etype_order:
        sub = df[df['error_type'] == et]
        if len(sub) == 0: continue
        em = ERROR_META.get(et, dict(label=et))
        row = dict(
            Session    = sess_label,
            Error_Type = et,
            N_Trials   = len(sub),
        )
        for wc, wl in zip(win_cols, win_labels):
            n_on = sub[wc].sum()
            row[wl] = f'{n_on}/{len(sub)} ({n_on/len(sub)*100:.0f}%)'
        rows.append(row)

summary_df = pd.DataFrame(rows)
display(summary_df)

summary_df.to_csv(COMBINED_DIR / 'error_stim_summary.csv', index=False)
print(f'\nSaved → {COMBINED_DIR / "error_stim_summary.csv"}')